# RAG Research Assistant — Demo Notebook

End-to-end walkthrough: ingest → embed → retrieve → generate.


In [ ]:
import sys; sys.path.insert(0, '../src')
from rag_pipeline import RAGPipeline

## 1. Instantiate the pipeline

In [ ]:
pipeline = RAGPipeline(
    embedding_model='all-MiniLM-L6-v2',
    generator_model='google/flan-t5-base',
    chunk_size=400,
    overlap=50,
    top_k=4,
)
print(pipeline)

## 2. Ingest a sample medical imaging corpus

In [ ]:
sample_text = """
U-Net is a convolutional neural network architecture designed for biomedical image segmentation.
It consists of an encoder (contracting path) that captures context and a decoder (expansive path)
that enables precise localization. Skip connections between encoder and decoder layers allow
the network to combine high-resolution spatial features with deep semantic features.

U-Net++ is an improved variant that replaces sparse skip connections with dense nested connections,
reducing the semantic gap between encoder and decoder feature maps. This architecture allows
better gradient flow and improved segmentation accuracy, especially for small objects.

SegNet is an encoder-decoder architecture for semantic pixel-wise labeling. Unlike U-Net,
SegNet uses pooling indices from max-pooling layers (instead of feature maps) for upsampling,
making it more memory-efficient.

Dice loss, also known as F1 loss, is commonly used in medical image segmentation to handle
class imbalance. It computes the overlap between prediction and ground truth directly, making
it more robust to datasets where background pixels vastly outnumber foreground pixels.

Intersection over Union (IoU), or Jaccard index, measures the overlap between predicted and
ground-truth masks. It is the primary evaluation metric in segmentation benchmarks such as
Pascal VOC, COCO, and medical imaging datasets.
"""

n = pipeline.ingest_text(sample_text, source_name='medical-imaging-notes')
print(f'Indexed {n} chunks')

## 3. Query the pipeline

In [ ]:
questions = [
    'What are skip connections and why are they important in U-Net?',
    'How does Dice loss handle class imbalance?',
    'What is the main difference between U-Net++ and U-Net?',
]

for q in questions:
    resp = pipeline.query(q)
    print(f'Q: {q}')
    print(f'A: {resp.answer}')
    print(f'  Top chunk similarity: {resp.retrieved_chunks[0].score:.3f}')
    print()

## 4. Inspect retrieved chunks

In [ ]:
resp = pipeline.query('How does SegNet differ from U-Net in upsampling?')
print(f'Answer: {resp.answer}\n')
for r in resp.retrieved_chunks:
    print(f'[Rank {r.rank+1}] Score={r.score:.3f} | Source={r.document.source}')
    print(r.document.text[:200], '...\n')